In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col

spark = SparkSession.builder \
    .appName("Spotify Bronze ETL") \
    .getOrCreate()

In [ ]:
dim_path = "../data/dim/"
fact_path = "../data/fact/"

dim_artists = spark.read.option("header", True).csv(dim_path + "dim_artists.csv")
dim_albums = (
    spark.read
    .option("header", True)
    .option("multiLine", True)  
    .option("quote", '"')       
    .option("escape", '"')      
    .option("mode", "PERMISSIVE")
    .csv(dim_path + "dim_albums.csv")
)
dim_genres  = spark.read.option("header", True).csv(dim_path + "dim_genres.csv")
dim_tracks = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv("../data/dim/dim_tracks.csv")
)
fact_tracks = spark.read.option("header", True).csv(fact_path + "fact_tracks.csv")

In [30]:
for df_name in ['dim_artists', 'dim_albums', 'dim_genres', 'dim_tracks', 'fact_tracks']:
    df = locals()[df_name]
    if 'ingest_date' not in df.columns:
        df = df.withColumn("ingest_date", current_timestamp())
    locals()[df_name] = df

In [31]:
bronze_path = "../delta_lake/bronze/"

dim_artists.write.mode("overwrite").parquet(bronze_path + "dim_artists")
dim_albums.write.mode("overwrite").parquet(bronze_path + "dim_albums")
dim_genres.write.mode("overwrite").parquet(bronze_path + "dim_genres")
dim_tracks.write.mode("overwrite").parquet(bronze_path + "dim_tracks")
fact_tracks.write.mode("overwrite").parquet(bronze_path + "fact_tracks")